<a href="https://colab.research.google.com/github/shubham-bari/Language-Models/blob/main/MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleExpert(nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class SimpleMoE(nn.Module):
    def __init__(self, d_model, d_hidden, num_experts):
        super().__init__()
        self.num_experts = num_experts

        # Experts
        self.experts = nn.ModuleList([
            SimpleExpert(d_model, d_hidden)
            for _ in range(num_experts)
        ])

        # Gating network
        self.gate = nn.Linear(d_model, num_experts)

    def forward(self, x):
        """
        x: (B, T, D)
        """
        B, T, D = x.shape

        # Compute gating scores
        gate_logits = self.gate(x)            # (B, T, E)
        gate_probs = F.softmax(gate_logits, dim=-1)

        # Top-1 expert selection
        top_expert = torch.argmax(gate_probs, dim=-1)  # (B, T)

        output = torch.zeros_like(x)

        # Route tokens
        for i in range(self.num_experts):
            mask = (top_expert == i)          # (B, T)
            if mask.sum() == 0:
                continue

            selected = x[mask]               # (num_tokens_i, D)
            out = self.experts[i](selected)  # (num_tokens_i, D)
            output[mask] = out

        return output


In [2]:
B, T, D = 4, 10, 64
x = torch.randn(B, T, D)

moe = SimpleMoE(d_model=64, d_hidden=128, num_experts=4)
y = moe(x)

print(y.shape)   # (4, 10, 64)


torch.Size([4, 10, 64])


In [3]:
moe

SimpleMoE(
  (experts): ModuleList(
    (0-3): 4 x SimpleExpert(
      (fc1): Linear(in_features=64, out_features=128, bias=True)
      (fc2): Linear(in_features=128, out_features=64, bias=True)
    )
  )
  (gate): Linear(in_features=64, out_features=4, bias=True)
)